In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list the files in the input directory

from subprocess import check_output
print(check_output(["ls", "../input"]).decode("utf8"))

# Any results you write to the current directory are saved as output.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

#Dataset

In [ ]:
train = pd.read_csv('../input/train.csv')
test = pd.read_csv('../input/test.csv')
train_valid_y = train['Survived']

###information of data

In [ ]:
train.info()
test.info()

In [ ]:
train.columns.values

In [ ]:
train.describe(include = ['O'])

In [ ]:
train.describe()

###View Categorical data

In [ ]:
def viewcat(df, cat, target):
    return df[[cat, target]].groupby(cat).mean().sort_values(by = target, ascending = False)

In [ ]:
viewcat(train, 'Pclass', 'Survived')

In [ ]:
viewcat(train, 'Sex', 'Survived')

In [ ]:
viewcat(train, 'SibSp', 'Survived')

In [ ]:
viewcat(train, 'Embarked', 'Survived')

###Plot Categorical data

In [ ]:
def plot_cat(df, cat, target):
    sns.barplot(df[cat], df[target])
    
plot_cat(train, 'Sex', 'Survived')

In [ ]:
plot_cat(train, 'Embarked', 'Survived')

###View Numeric data

In [ ]:
def plot_num(df, var, target, **kwargs):
    row = kwargs.get('row', None)
    col = kwargs.get('col',  None)
    grid = sns.FacetGrid(df, hue = target, row = row, col = col)
    grid.map(sns.distplot, var)
    grid.add_legend()

plot_num(train, 'Fare', 'Survived')

In [ ]:
grid = sns.FacetGrid(train, hue = 'Survived', col = 'Embarked')
grid.map(plt.hist, 'Age', alpha = .5, bins = 20)
grid.add_legend()

#Feature engineering

In [ ]:
train.drop('Survived', axis = 1, inplace = True)

In [ ]:
full = pd.concat([train,test])

In [ ]:
full.info()

In [ ]:
full.describe(include = ['O'])

###Categorical Data: get_dummies

In [ ]:
full['Embarked'].fillna('S', inplace = True)

In [ ]:
full.info()

In [ ]:
Sex = pd.get_dummies(full['Sex'])

In [ ]:
Embarked = pd.get_dummies(full['Embarked'])

In [ ]:
full['Title'] = full['Name'].map(lambda name: name.split(',')[1].split('.')[0].strip())
pd.crosstab(full['Title'], full['Sex'])

In [ ]:
full['Title'] = full['Title'].replace(['Capt', 'Col', 'Don', 'Dona', 'Dr', 'Jonkheer', 'Lady', 'Major', 'Rev', 'the Countess'], 'Rare')
full['Title'] = full['Title'].replace(['Mlle', 'Mme', 'Ms'], 'Mrs')
full['Title'].unique()
Title = pd.get_dummies(full['Title'])

In [ ]:
Title

In [ ]:
full.drop(['Title', 'Embarked', 'Name', 'Sex', 'PassengerId', 'Cabin'], axis = 1, inplace = True)

In [ ]:
full

In [ ]:
full['Family'] = full['SibSp'] + full['Parch']

In [ ]:
full.info()

###Numerical Data: fillna and cut

In [ ]:
full['Fare'] = full['Fare'].fillna(full['Fare'].mean())

In [ ]:
full.info()

In [ ]:
full.drop(['SibSp', 'Parch'], inplace = True, axis = 1)

In [ ]:
full = pd.concat([full,Sex, Embarked, Title], axis = 1)

In [ ]:
full

In [ ]:
sns.heatmap(full.corr())

In [ ]:
for i in range(3):
    guess = full[full['Pclass'] == i+1]['Age'].median()
    full.loc[(full['Age'].isnull()) & (full['Pclass'] == i+1), 'Age'] = guess

In [ ]:
full.info()

In [ ]:
full.drop('Ticket', axis = 1, inplace = True)

In [ ]:
full

#Model

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression  
from sklearn.naive_bayes import GaussianNB  
from sklearn.metrics import accuracy_score

In [ ]:
train_valid_x = full[0:891]
test_x = full[891:]
X_train, X_valid, y_train, y_valid = train_test_split(train_valid_x, train_valid_y, random_state = 0) 

In [ ]:
X_train.info()

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_train_valid_scaled = scaler.transform(train_valid_x)
y_train_valid = train_valid_y

###Model class

In [ ]:
class Model(object):
    def __init__(self, clf, seed = 0, **params):
        params['random_state'] = seed
        self.clf = clf(**params)
    def fit(self, x, y):
        return self.clf.fit(x, y)
    def predict(self, x):
        return self.clf.predict(x)
    def score(self, x, y):
        y_predict = self.clf.predict(x)
        return accuracy_score(y, y_predict)
    
        
        

In [ ]:
rf = Model(clf = RandomForestClassifier)
gb = Model(clf = GradientBoostingClassifier)
mlp = Model(clf = MLPClassifier)
svc = Model(clf = SVC)
xgb = Model(clf = XGBClassifier)
clf1 = LogisticRegression(random_state=0)  
clf2 = RandomForestClassifier(random_state=0)  
clf3 = GaussianNB()
eclf = VotingClassifier(estimators=[('lr', clf1), ('rf', clf2), ('gnb', clf3)], voting='hard') 
eclf.fit(X_train_scaled, y_train)
rf.fit(X_train_scaled, y_train)
gb.fit(X_train_scaled, y_train)
mlp.fit(X_train_scaled, y_train)
svc.fit(X_train_scaled, y_train)
xgb.fit(X_train_scaled, y_train)

In [ ]:
from sklearn.model_selection import cross_val_score
rf_scores = cross_val_score(RandomForestClassifier(), 
                            X_train_valid_scaled, y_train_valid, scoring = 'accuracy',
                            cv= 5)
gb_scores = cross_val_score(GradientBoostingClassifier(), X_train_valid_scaled, y_train_valid, scoring = 'accuracy'
                           ,cv = 5)
#mlp_scores = cross_val_score(MLPClassifier(), X_train_valid_scaled, y_train_valid, scoring = 'accuracy'
                           #, cv = 3)
svc_scores = cross_val_score(SVC(), X_train_valid_scaled, y_train_valid, scoring = 'accuracy'
                           , cv = 5)
xgb_scores = cross_val_score(XGBClassifier(), X_train_valid_scaled, y_train_valid, scoring = 'accuracy'
                           , cv = 5)
elf_scores = cross_val_score(eclf, X_train_valid_scaled, y_train_valid, scoring = 'accuracy'
                           , cv = 5)

In [ ]:
rf_scores.mean()

In [ ]:
gb_scores.mean()

In [ ]:
svc_scores.mean()

In [ ]:
xgb_scores.mean()

In [ ]:
elf_scores.mean()

In [ ]:
rf_score = rf.score(X_valid_scaled, y_valid)
gb_score = gb.score(X_valid_scaled, y_valid)
mlp_score = mlp.score(X_valid_scaled, y_valid)
svc_score = svc.score(X_valid_scaled, y_valid)
xgb_score = xgb.score(X_valid_scaled, y_valid)
elf_y_predict = eclf.predict(X_valid_scaled)
elf_score = accuracy_score(y_valid, elf_y_predict)

In [ ]:
print(rf_score, gb_score, mlp_score, svc_score, xgb_score, elf_score)

In [ ]:
X_test_scaled = scaler.transform(test_x)

In [ ]:
y_predict = xgb.predict(X_test_scaled)

In [ ]:
y_predict

In [ ]:
PassengerId = test['PassengerId']
submission = pd.DataFrame({'PassengerId': PassengerId, 'Survived': y_predict})

In [ ]:
submission.to_csv('submission.csv', index = False)